# 01 — Data Exploration & Preprocessing

**Project:** Clinical NLP — Drug Review Sentiment Classification with XAI & UQ  
**Author:** Garcia Wa Nnam  
**Dataset:** UCI Drug Review Dataset (Kallumadi & Grer, 2018)  

---

## Objective

This notebook covers the complete exploratory data analysis (EDA) phase of the pipeline:

1. Load and inspect the raw drug review data
2. Analyse class distribution and identify imbalance
3. Explore text characteristics (review length, vocabulary)
4. Visualise the most common drug conditions and review sentiment patterns
5. Apply preprocessing and save clean datasets for downstream modelling

---

## Clinical Relevance

Drug reviews written by patients represent a form of **patient-reported outcome (PRO)** data — an increasingly important signal in health technology assessment (HTA). Organisations like NICE routinely consider patient experience evidence when evaluating treatments. Building NLP pipelines that can extract structured sentiment from unstructured patient text is therefore a high-value capability in health data science.

> **Note on ethics:** This dataset is publicly available and anonymised. Reviews contain no patient-identifiable information. All modelling decisions in this project consider the clinical implications of misclassification.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

from utils import clean_text, binarise_rating, load_and_prepare, set_seed

set_seed(42)

# Consistent plot style throughout project
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')

## 2. Load Raw Data

In [ ]:
TRAIN_PATH = '../data/drugsComTrain_raw.tsv'
TEST_PATH  = '../data/drugsComTest_raw.tsv'

df_train_raw = pd.read_csv(TRAIN_PATH, sep='\t', on_bad_lines='skip')
df_test_raw  = pd.read_csv(TEST_PATH,  sep='\t', on_bad_lines='skip')

print(f'Train shape : {df_train_raw.shape}')
print(f'Test shape  : {df_test_raw.shape}')
df_train_raw.head(3)

In [ ]:
# Inspect data types and missing values
print('--- Training Data Info ---')
print(df_train_raw.info())
print('\n--- Missing Values ---')
print(df_train_raw.isnull().sum())

## 3. Rating Distribution

Ratings are on a 1–10 scale. We will binarise these using a threshold of **≥7 = Positive**,  
reflecting clinically meaningful patient satisfaction (consistent with PRO literature).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title) in zip(axes, [(df_train_raw, 'Training Set'), (df_test_raw, 'Test Set')]):
    counts = df['rating'].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette('muted'), edgecolor='white')
    ax.axvline(x=6.5, color='red', linestyle='--', linewidth=1.5, label='Threshold (≥7 = Positive)')
    ax.set_title(f'Rating Distribution — {title}', fontweight='bold')
    ax.set_xlabel('Rating (1–10)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

plt.tight_layout()
plt.savefig('../outputs/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to outputs/')

## 4. Class Imbalance Analysis

Understanding class balance is essential before any modelling step.  
Imbalanced classes can cause classifiers to be biased toward the majority class —  
in a clinical context, this could mean systematically under-detecting negative patient experiences.

In [ ]:
df_train_raw.dropna(subset=['review','rating'], inplace=True)
df_test_raw.dropna(subset=['review','rating'], inplace=True)

df_train_raw['label'] = df_train_raw['rating'].apply(binarise_rating)
df_test_raw['label']  = df_test_raw['rating'].apply(binarise_rating)

for name, df in [('Train', df_train_raw), ('Test', df_test_raw)]:
    counts = df['label'].value_counts()
    pct    = df['label'].value_counts(normalize=True) * 100
    print(f'\n{name} Set:')
    print(f'  Negative (0): {counts[0]:>6,}  ({pct[0]:.1f}%)')
    print(f'  Positive (1): {counts[1]:>6,}  ({pct[1]:.1f}%)')
    print(f'  Imbalance ratio: 1:{counts[1]/counts[0]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels_map = {0: 'Negative', 1: 'Positive'}

for ax, (df, title) in zip(axes, [(df_train_raw, 'Train'), (df_test_raw, 'Test')]):
    counts = df['label'].value_counts().rename(labels_map)
    ax.pie(counts, labels=counts.index, autopct='%1.1f%%',
           colors=['#e07b7b', '#6baed6'], startangle=90,
           wedgeprops={'edgecolor':'white', 'linewidth':2})
    ax.set_title(f'Class Distribution — {title}', fontweight='bold')

plt.suptitle('Binary Sentiment Labels (Threshold ≥7)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Text Length Analysis

Transformer models have a maximum token limit (typically 512 tokens for BERT-based models).  
Understanding review length helps us choose an appropriate truncation strategy.

In [ ]:
df_train_raw['review_clean'] = df_train_raw['review'].apply(clean_text)
df_train_raw['word_count']   = df_train_raw['review_clean'].apply(lambda x: len(x.split()))

print('Review length statistics (word count):')
print(df_train_raw['word_count'].describe().round(1))

pct_over_128 = (df_train_raw['word_count'] > 128).mean() * 100
pct_over_256 = (df_train_raw['word_count'] > 256).mean() * 100
print(f'\n{pct_over_128:.1f}% of reviews exceed 128 words')
print(f'{pct_over_256:.1f}% of reviews exceed 256 words')
print('→ Truncation at 128 tokens is a reasonable trade-off for efficiency.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_train_raw['word_count'].clip(upper=400), bins=60,
        color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(128, color='red', linestyle='--', linewidth=2, label='128-word truncation point')
ax.axvline(256, color='orange', linestyle='--', linewidth=2, label='256-word truncation point')
ax.set_xlabel('Word Count (clipped at 400)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Review Lengths — Training Set', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/review_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Top Drug Conditions

In [ ]:
top_conditions = (df_train_raw['condition']
                  .dropna()
                  .str.strip()
                  .value_counts()
                  .head(20))

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_conditions.index[::-1], top_conditions.values[::-1],
               color=sns.color_palette('Blues_d', 20))
ax.set_xlabel('Number of Reviews')
ax.set_title('Top 20 Drug Conditions by Review Count', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/top_conditions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Word Clouds — Positive vs Negative Reviews

In [ ]:
pos_text = ' '.join(df_train_raw[df_train_raw['label']==1]['review_clean'].sample(2000, random_state=42))
neg_text = ' '.join(df_train_raw[df_train_raw['label']==0]['review_clean'].sample(2000, random_state=42))

stopwords_extra = {'drug', 'medication', 'mg', 'dose', 'take', 'taking',
                   'also', 'one', 'would', 'like', 'get', 'got', 'year',
                   'month', 'day', 'week', 'time', 'use', 'used'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, text, title, cmap in zip(
    axes,
    [pos_text, neg_text],
    ['Positive Reviews (Rating ≥ 7)', 'Negative Reviews (Rating < 7)'],
    ['Blues', 'Reds']
):
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=cmap, stopwords=stopwords_extra,
                   max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontweight='bold', fontsize=13)

plt.suptitle('Word Clouds by Sentiment Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/wordclouds_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Cleaned Datasets

Saving preprocessed DataFrames so downstream notebooks can load clean data directly,  
rather than repeating cleaning steps — a good data engineering practice.

In [ ]:
df_train_raw['review_clean'] = df_train_raw['review'].apply(clean_text)
df_test_raw['review_clean']  = df_test_raw['review'].apply(clean_text)
df_test_raw['label']         = df_test_raw['rating'].apply(binarise_rating)

cols_save = ['review_clean', 'label', 'rating', 'condition', 'drugName']
df_train_clean = df_train_raw[[c for c in cols_save if c in df_train_raw.columns]]
df_test_clean  = df_test_raw[[c for c in cols_save if c in df_test_raw.columns]]

df_train_clean.to_csv('../data/train_clean.csv', index=False)
df_test_clean.to_csv('../data/test_clean.csv', index=False)

print(f'Saved train_clean.csv : {df_train_clean.shape}')
print(f'Saved test_clean.csv  : {df_test_clean.shape}')

---
## Summary

| Aspect | Finding |
|---|---|
| Train size | 161,297 reviews |
| Test size | 53,766 reviews |
| Class balance | ~65% Positive / 35% Negative |
| Median review length | ~87 words |
| Truncation strategy | 128 tokens (covers ~75% of reviews fully) |
| Missing data | Minimal (<1% in condition column) |

**Key insight:** The dataset has a moderate class imbalance (~65/35). This will be accounted for using class-weighted loss functions in the baseline model, and we will report both overall accuracy and per-class F1 to avoid misleading aggregate metrics.

→ Proceed to `02_baseline_model.ipynb`